In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df = pd.read_csv("../data/raw/train.csv")

In [5]:
df["IsFemale"] = df["Sex"] == "female"
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

# Build Title and finish ALL text-based work on it first
df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.")
df.loc[~df["Title"].isin(["Mr", "Miss", "Mrs", "Master"]), "Title"] = "Rare"

# Fill Age while Title is still plain text (needed for groupby)
df["Age"] = df["Age"].fillna(df.groupby("Title")["Age"].transform("median"))

# Fill Embarked before encoding it
df["Embarked"] = df["Embarked"].fillna("S")

# NOW one-hot encode both, once everything that needed the plain versions is done
df = pd.get_dummies(df, columns=["Title"], prefix="Title")
df = pd.get_dummies(df, columns=["Embarked"], prefix="Embarked")

# HasCabin doesn't depend on ordering relative to the above
df["HasCabin"] = df["Cabin"].notna()

In [7]:
feature_cols = [
    "IsFemale", "Pclass", "Age", "Fare", "FamilySize", "HasCabin",
    "Title_Master", "Title_Miss", "Title_Mr", "Title_Mrs", "Title_Rare",
    "Embarked_C", "Embarked_Q", "Embarked_S"
]
X = df[feature_cols]
y = df["Survived"]

# 11. Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [9]:
X.isna().sum()

IsFemale        0
Pclass          0
Age             0
Fare            0
FamilySize      0
HasCabin        0
Title_Master    0
Title_Master    0
Title_Miss      0
Title_Miss      0
Title_Mr        0
Title_Mr        0
Title_Mrs       0
Title_Mrs       0
Title_Rare      0
Embarked_C      0
Embarked_Q      0
Embarked_S      0
dtype: int64

In [10]:
X.shape

(891, 18)

In [12]:
X.head()

,IsFemale,Pclass,Age,Fare,FamilySize,HasCabin,Title_Master,Title_Master,Title_Miss,Title_Miss,Title_Mr,Title_Mr,Title_Mrs,Title_Mrs,Title_Rare,Embarked_C,Embarked_Q,Embarked_S
0,False,3,22.0,7.2500,2,False,False,False,False,False,True,True,False,False,False,False,False,True
1,True,1,38.0,71.2833,2,True,False,False,False,False,False,False,True,True,False,True,False,False
2,True,3,26.0,7.9250,1,False,False,False,True,True,False,False,False,False,False,False,False,True
3,True,1,35.0,53.1000,2,True,False,False,False,False,False,False,True,True,False,False,False,True
4,False,3,35.0,8.0500,1,False,False,False,False,False,True,True,False,False,False,False,False,True
